2 序列模型

2.1 理论计算题


给定字符序列 "ababc"，采用一阶马尔可夫模型，使用拉普拉斯平滑（加1平滑）估计条件概率。

词汇表: {a, b, c}

统计转移次数:

首先统计所有相邻转移（一阶）：

a → b: 出现2次（位置1: a→b, 位置3: a→b）

b → a: 出现1次（位置2: b→a）

b → c: 出现1次（位置4: b→c）

转移计数矩阵（行=前一个词，列=后一个词）：


前\后	a	b	c

a	0	2	0

b	1	0	1

c	0	0	0

词汇表大小 V = 3

拉普拉斯平滑公式：
p(xt∣xt−1)= （count(xt−1,xt)+1）/ （count(xt−1)+V）

1. p(a|b):

count(b, a) = 1

count(b) = 2 (b出现2次)

p(a|b) = (1 + 1) / (2 + 3) = 2/5 = 0.4

2. p(c|b):

count(b, c) = 1

count(b) = 2

p(c|b) = (1 + 1) / (2 + 3) = 2/5 = 0.4

答案: p(a|b) = 0.4, p(c|b) = 0.4

2.2 编程题

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    预处理文本，构建词汇表并生成特征序列和标签
    
    Args:
        text: 输入文本字符串
        n: 滑动窗口大小（特征序列长度）
    
    Returns:
        vocab: 词汇表字典 {词: ID}
        features: 特征列表，每个特征是一个长度为n的词列表
        labels: 标签列表
    """
    # 1. 转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    
    # 2. 按空格分词
    words = text.split()
    
    if len(words) <= n:
        # 如果词数不够，返回空列表
        return {}, [], []
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(words)
    # 按频率降序排序，频率相同按字母顺序
    sorted_words = sorted(word_counts.items(), key=lambda x: (-x[1], x[0]))
    vocab = {word: idx for idx, (word, _) in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成特征序列和标签
    features = []
    labels = []
    
    for i in range(len(words) - n):
        features.append(words[i:i+n])
        labels.append(words[i+n])
    
    return vocab, features, labels

# 测试示例
if __name__ == "__main__":
    text = "The time machine"
    n = 2
    vocab, features, labels = preprocess_text(text, n)
    
    print("词汇表:", vocab)
    print("特征:", features)
    print("标签:", labels)
    
    # 额外测试
    print("\n--- 额外测试 ---")
    text2 = "Hello world hello python programming deep learning"
    n = 3
    vocab2, features2, labels2 = preprocess_text(text2, n)
    print("词汇表:", vocab2)
    print("特征:", features2)
    print("标签:", labels2)

词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征: [['the', 'time']]
标签: ['machine']

--- 额外测试 ---
词汇表: {'hello': 0, 'deep': 1, 'learning': 2, 'programming': 3, 'python': 4, 'world': 5}
特征: [['hello', 'world', 'hello'], ['world', 'hello', 'python'], ['hello', 'python', 'programming'], ['python', 'programming', 'deep']]
标签: ['python', 'programming', 'deep', 'learning']


3 循环神经网络

3.1 理论计算题

线性RNN定义:

ht=Whh* ht−1 +Whx *xt
​

ot=Woh*ht
​ 

L= 1/2 ∑ t=1——>T (ot−yt) ^2
 
通过时间反向传播（BPTT）推导:

损失对 W hh  的梯度：

∂L/∂Whh =∑ t=1——>T ∂L/∂ot ⋅ ∂ot/∂ht ⋅ ∂ht/∂Whh

​其中：


∂L/∂ot =ot −yt


​∂ot/∂ht =Woh
​

对于 ∂ht/∂Whh ，需要考虑所有时间步的依赖：


∂ht/∂Whh =∑ k=1——>t (∏ j=k+1——>t Whh )⋅ ∂hk/∂Whh

​
其中 ∂hk/∂Whh =h k−1 （因为 h k =W hh*h k−1 +W hx * x k ）


因此：


∂L/∂W hh =∑ t=1——>T (o t −y t )W oh ∑ k=1——>t W(hh——>t−k)* h k−1
​
梯度消失/爆炸条件:

梯度爆炸: 当 W hh  的最大特征值 > 1 时，
W hh——>t−k  随着 t−k 增大而指数增长，导致梯度爆炸

梯度消失: 当 W hh  的最大特征值 < 1 时，
W hh——>t−k  随着 t−k 增大而指数衰减到0，导致梯度消失

3.2 编程题

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单元前向传播
    
    Args:
        x_t: 当前输入，形状 (batch_size, input_size)
        h_prev: 上一隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏权重，形状 (input_size, hidden_size)
        W_hh: 隐藏到隐藏权重，形状 (hidden_size, hidden_size)
        b_h: 偏置，形状 (hidden_size,)
    
    Returns:
        h_t: 当前隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存用于反向传播
    """
    # 计算隐藏状态
    h_t = np.tanh(np.dot(x_t, W_hx) + np.dot(h_prev, W_hh) + b_h)
    
    # 缓存中间变量用于反向传播
    cache = (x_t, h_prev, h_t, W_hx, W_hh)
    
    return h_t, cache

def rnn_cell_backward(dh_next, cache):
    """
    RNN单元单步反向传播
    
    Args:
        dh_next: 上游梯度，即损失对h_t的梯度，形状 (batch_size, hidden_size)
        cache: 前向传播缓存
    
    Returns:
        dx_t: 损失对x_t的梯度，形状 (batch_size, input_size)
        dh_prev: 损失对h_prev的梯度，形状 (batch_size, hidden_size)
        dW_hx: 损失对W_hx的梯度，形状 (input_size, hidden_size)
        dW_hh: 损失对W_hh的梯度，形状 (hidden_size, hidden_size)
        db_h: 损失对b_h的梯度，形状 (hidden_size,)
    """
    x_t, h_prev, h_t, W_hx, W_hh = cache
    
    batch_size, hidden_size = h_t.shape
    
    # tanh的导数: dtanh = 1 - tanh^2
    dh = dh_next * (1 - h_t**2)
    
    # 计算各个梯度
    dx_t = np.dot(dh, W_hx.T)
    dh_prev = np.dot(dh, W_hh.T)
    dW_hx = np.dot(x_t.T, dh)
    dW_hh = np.dot(h_prev.T, dh)
    db_h = np.sum(dh, axis=0)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试代码
if __name__ == "__main__":
    np.random.seed(42)
    
    batch_size = 3
    input_size = 4
    hidden_size = 5
    
    # 随机初始化
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(hidden_size)
    
    # 前向传播
    h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("h_t shape:", h_t.shape)
    print("h_t:\n", h_t)
    
    # 反向传播
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)
    
    print("\n梯度形状:")
    print("dx_t:", dx_t.shape)
    print("dh_prev:", dh_prev.shape)
    print("dW_hx:", dW_hx.shape)
    print("dW_hh:", dW_hh.shape)
    print("db_h:", db_h.shape)

h_t shape: (3, 5)
h_t:
 [[ 0.37757047 -0.85065863 -0.99999996 -0.95887056  0.60632966]
 [-0.99893487 -0.99615252 -0.99992555  0.99878199 -0.02801693]
 [ 0.5842284   0.42268155 -0.9914534  -0.70369631 -0.77853071]]

梯度形状:
dx_t: (3, 4)
dh_prev: (3, 5)
dW_hx: (4, 5)
dW_hh: (5, 5)
db_h: (5,)


4 高级循环神经网络


4.1 理论计算题


深度双向RNN参数计算:

L层，每层隐藏单元数H

输入维度D

输出维度O

每层参数（以第l层为例）：

前向RNN:

输入到隐藏: D_l × H（其中D_l = D（第一层），D_l = 2H（后续层，因为输入是前一层拼接的前向和后向输出））

隐藏到隐藏: H × H

偏置: H

后向RNN:

输入到隐藏: D_l × H

隐藏到隐藏: H × H

偏置: H

每层总计（对于第l层）:

第1层: 2 × (D×H + H×H + H) = 2DH + 2H² + 2H

第l层 (l>1): 2 × (2H×H + H×H + H) = 6H² + 2H

总参数:

Total=(2DH+2H ^2 +2H)+(L−1)(6H ^2 +2H)

忽略输出层投影，即不包括输出层参数。

4.2 编程题

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器
    使用torch.nn.RNN实现
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BidirectionalRNNEncoder, self).__init__()
        self.hidden_dim = hidden_dim
        
        # 使用双向RNN
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=False,  # 输入形状: (seq_len, batch, input_dim)
            bidirectional=True
        )
    
    def forward(self, X):
        """
        Args:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        Returns:
            outputs: 每个时间步拼接后的隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        # rnn输出: outputs (seq_len, batch, 2*hidden_dim), 
        # hidden_state (num_layers*2, batch, hidden_dim)
        outputs, hidden_state = self.rnn(X)
        
        # 获取最终时间步的隐藏状态
        # 对于双向RNN，最终状态是最后一层的前向和后向的拼接
        # hidden_state形状: (num_layers * num_directions, batch, hidden_dim)
        # 最后一层的前向: hidden_state[-2], 后向: hidden_state[-1]
        final_forward = hidden_state[-2, :, :]  # (batch, hidden_dim)
        final_backward = hidden_state[-1, :, :]  # (batch, hidden_dim)
        final_state = torch.cat([final_forward, final_backward], dim=1)  # (batch, 2*hidden_dim)
        
        return outputs, final_state


class BidirectionalRNNEncoderManual(nn.Module):
    """
    手动实现的双向RNN编码器（使用RNN单元）
    """
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BidirectionalRNNEncoderManual, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 前向RNN层
        self.rnn_forward = nn.RNNCell(input_dim, hidden_dim)
        # 后向RNN层
        self.rnn_backward = nn.RNNCell(input_dim, hidden_dim)
        
        # 如果是多层，需要更多的RNN层
        # 这里为了简化，只实现单层
    
    def forward(self, X):
        """
        Args:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        Returns:
            outputs: 每个时间步拼接后的隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            final_state: 最终时间步的拼接隐藏状态，形状 (batch, 2*hidden_dim)
        """
        seq_len, batch_size, _ = X.shape
        
        # 初始化隐藏状态
        h_forward = torch.zeros(batch_size, self.hidden_dim, device=X.device)
        h_backward = torch.zeros(batch_size, self.hidden_dim, device=X.device)
        
        forward_outputs = []
        backward_outputs = []
        
        # 前向传播（从前往后）
        for t in range(seq_len):
            h_forward = self.rnn_forward(X[t], h_forward)
            forward_outputs.append(h_forward)
        
        # 后向传播（从后往前）
        for t in range(seq_len - 1, -1, -1):
            h_backward = self.rnn_backward(X[t], h_backward)
            backward_outputs.append(h_backward)
        
        # 反转后向输出使其按时间顺序
        backward_outputs = list(reversed(backward_outputs))
        
        # 拼接前向和后向输出
        outputs = []
        for f, b in zip(forward_outputs, backward_outputs):
            outputs.append(torch.cat([f, b], dim=1))
        
        # 转换为张量
        outputs = torch.stack(outputs, dim=0)  # (seq_len, batch, 2*hidden_dim)
        
        # 最终状态：最后一个时间步的拼接
        final_state = outputs[-1]  # (batch, 2*hidden_dim)
        
        return outputs, final_state


# 测试代码
if __name__ == "__main__":
    # 参数设置
    seq_len = 5
    batch_size = 3
    input_dim = 10
    hidden_dim = 8
    
    # 使用PyTorch内置RNN
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
    X = torch.randn(seq_len, batch_size, input_dim)
    outputs, final_state = encoder(X)
    
    print("使用PyTorch内置RNN:")
    print("outputs shape:", outputs.shape)  # (seq_len, batch, 2*hidden_dim)
    print("final_state shape:", final_state.shape)  # (batch, 2*hidden_dim)
    
    # 手动实现
    encoder_manual = BidirectionalRNNEncoderManual(input_dim, hidden_dim)
    outputs_manual, final_state_manual = encoder_manual(X)
    
    print("\n手动实现:")
    print("outputs shape:", outputs_manual.shape)
    print("final_state shape:", final_state_manual.shape)

使用PyTorch内置RNN:
outputs shape: torch.Size([5, 3, 16])
final_state shape: torch.Size([3, 16])

手动实现:
outputs shape: torch.Size([5, 3, 16])
final_state shape: torch.Size([3, 16])


5 嵌入向量


5.1 理论计算题


Skip-gram with Negative Sampling (SGNS):

给定中心词 w c  和上下文词 w o ，负采样损失函数：

L=−logσ(v wc ⋅u wo )−∑ k=1——>K E wn ∼P n(w) [logσ(−v wc ⋅u wn )]

其中 σ(x)= 1/(1+e ^(−x))  是sigmoid函数。

完整目标函数（对于一个中心词-上下文词对）：

L=−logσ(v c ⋅u o )−∑ k=1——>K logσ(−v c ⋅u nk )

负样本采样：

从噪声分布 P n (w) 中采样 K 个负样本。常用的噪声分布是词频的3/4次方（Mikolov et al.）：

P n (w)=  freq(w) ^(3/4)/∑ w 'freq(w') ^(3/4)


这样采样会提高低频词被选中的概率。

5.2 编程题

In [2]:
import numpy as np

def cbow_forward(context_indices, W, W_out, target_idx):
    """
    CBOW模型前向传播和损失计算
    
    Args:
        context_indices: 一批上下文词索引，形状 (batch_size, context_size)
        W: 输入权重矩阵，形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
        target_idx: 目标中心词索引，形状 (batch_size,)
    
    Returns:
        loss: 交叉熵损失值
        probs: 输出概率分布，形状 (batch_size, V)
        hidden: 平均上下文向量（隐藏层），形状 (batch_size, d)
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 1. 获取上下文词向量
    # context_indices: (batch_size, context_size) -> 每个样本取对应词向量
    # 得到 (batch_size, context_size, d)
    context_embeddings = W[context_indices]  # (batch_size, context_size, d)
    
    # 2. 计算平均上下文向量作为隐藏层
    hidden = np.mean(context_embeddings, axis=1)  # (batch_size, d)
    
    # 3. 计算输出分数 (logits)
    scores = np.dot(hidden, W_out)  # (batch_size, V)
    
    # 4. 计算softmax概率分布
    # 使用数值稳定性技巧：减去最大值
    scores_shifted = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores_shifted)
    probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)  # (batch_size, V)
    
    # 5. 计算交叉熵损失
    # 选择目标词对应的概率
    batch_indices = np.arange(batch_size)
    target_probs = probs[batch_indices, target_idx]  # (batch_size,)
    
    # 防止log(0)
    target_probs = np.clip(target_probs, 1e-10, 1.0)
    loss = -np.mean(np.log(target_probs))
    
    return loss, probs, hidden

# 测试代码
if __name__ == "__main__":
    np.random.seed(42)
    
    # 参数设置
    V = 10  # 词汇表大小
    d = 4   # 嵌入维度
    context_size = 3
    batch_size = 2
    
    # 随机初始化
    W = np.random.randn(V, d) * 0.01
    W_out = np.random.randn(d, V) * 0.01
    
    # 生成随机数据
    context_indices = np.random.randint(0, V, size=(batch_size, context_size))
    target_idx = np.random.randint(0, V, size=batch_size)
    
    print("上下文索引 (batch_size, context_size):", context_indices.shape)
    print("目标索引 (batch_size,):", target_idx.shape)
    
    # 前向传播
    loss, probs, hidden = cbow_forward(context_indices, W, W_out, target_idx)
    
    print("\n损失值:", loss)
    print("输出概率分布形状:", probs.shape)
    print("隐藏层形状:", hidden.shape)
    print("\n输出概率分布 (第一个样本):", probs[0])
    print("目标词概率:", probs[0, target_idx[0]])

上下文索引 (batch_size, context_size): (2, 3)
目标索引 (batch_size,): (2,)

损失值: 2.3026460756235387
输出概率分布形状: (2, 10)
隐藏层形状: (2, 4)

输出概率分布 (第一个样本): [0.09999734 0.09999432 0.10000912 0.10001016 0.10000688 0.09998799
 0.09999991 0.09997957 0.09999172 0.10002298]
目标词概率: 0.09998799065697155


6 注意力机制

6.1 理论计算题

给定：

Q∈R ^(2×4)
 
K∈R ^(3X4)

V∈R ^(3X5)

d k=4

步骤1：计算得分矩阵

Score= QK ^T/根号d k=QK ^T /2

设 Q=[ q 11 q 12 q 13 q 14
      
       q 21 q 22 q 23 q 24 ]，

K=  [ k 11  k 12  k 13  k 14

​
      k 21  k 22  k 23  k 24
      
​
      k 31  k 32  k 33  k 34 ]
​
 
则 QK ^T ∈R ^(2×3) ，每个元素为：

(QK ^T ) ij = m=1——>4 ∑ Q im * K jm
​
 
因此：

Score= 1/2 [   ∑ m q 1m k 1m     ∑ m q 1m k 2m     ∑ m q 1m k 3m

​
                 ∑ m q 2m k 1m     ∑ m q 2m k 2m    ∑ m q 2m k 3m  ]

步骤2：Softmax（按行）

对Score矩阵的每一行应用softmax：

A ij= (e ^Score ij)/(∑ l=1——>3 e ^Score il) (i=1,2;j=1,2,3)

得到注意力权重矩阵 

A∈R ^(2×3) 。

步骤3：加权求和

Output=A×V∈R ^(2X5)
 
Output ij = l=1——>3 ∑ A il * V lj
​
 
具体数值示例：

设具体数值（简化）：

Q =[  1  0  1  0

      0  1  0  1  ]


K =  [  1  1  0  0

        0  0  1  1

        1  0  1  0  ]


V  = [  1  2  3  4  5

        6  7  8  9  10

        11  12  13  14  15  ]

​
​计算 QK^ T ：

(1,1): 1×1+0×1+1×0+0×0=1

(1,2): 1×0+0×0+1×1+0×1=1

(1,3): 1×1+0×0+1×1+0×0=2

(2,1): 0×1+1×1+0×0+1×0=1

(2,2): 0×0+1×0+0×1+1×1=1

(2,3): 0×1+1×0+0×1+1×0=0

QK ^T =[   1  1  2

​
           1  1  0   ]


Score= 1/2 [  1  1  2

              1  1  0  ]

     =  [  0.5  0.5  1

           0.5  0.5  0  ]

​
Softmax：
  
​第1行：
e ^0.5 =1.6487,e ^0.5 =1.6487,e ^1 =2.7183，总和 6.0157

A 11 =0.2741,A 12 =0.2741,A 13 =0.4518

第2行：

e ^0.5 =1.6487,e ^0.5 =1.6487,e ^0 =1，总和 4.2974


A 21 =0.3837,A 22 =0.3837,A 23 =0.2326

A  =  [   0.2741   0.2741   0.4518

          0.3837   0.3837   0.2326   ]

加权求和：

输出第1行：
0.2741×V 1 +0.2741×V 2 +0.4518×V 3
​= [0.2741+1.6446+4.9698,0.5482+1.9187+5.4216,0.8223+2.1928+5.8734,1.0964+2.4669+6.3252,1.3705+2.741+6.777]

= [6.8885,7.8885,8.8885,9.8885,10.8885]

输出第2行：
0.3837×V 1 +0.3837×V 2 +0.2326×V 3
​
= [0.3837+2.3022+2.5586,0.7674+2.6859+2.7912,1.1511+3.0696+3.0238,1.5348+3.4533+3.2564,1.9185+3.837+3.489]

= [5.2445,6.2445,7.2445,8.2445,9.2445]

Output = [  6.8885   7.8885   8.8885   9.8885   10.8885

            5.2445   6.2445   7.2445   8.2445   9.2445   ]




6.2 编程题

In [3]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制
    
    参数:
        d_model: 模型维度
        num_heads: 注意力头数
    """
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.d_v = d_model // num_heads
        
        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # 最终输出层
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, X):
        """
        前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, d_model)
        
        返回:
            output: 输出，形状 (seq_len, batch, d_model)
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影得到Q, K, V
        Q = self.W_q(X)  # (seq_len, batch, d_model)
        K = self.W_k(X)  # (seq_len, batch, d_model)
        V = self.W_v(X)  # (seq_len, batch, d_model)
        
        # 2. 重塑为多头形式
        # (seq_len, batch, d_model) -> (seq_len, batch, num_heads, d_k)
        Q = Q.view(seq_len, batch_size, self.num_heads, self.d_k)
        K = K.view(seq_len, batch_size, self.num_heads, self.d_k)
        V = V.view(seq_len, batch_size, self.num_heads, self.d_v)
        
        # 交换维度: (seq_len, batch, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        Q = Q.permute(1, 2, 0, 3)
        K = K.permute(1, 2, 0, 3)
        V = V.permute(1, 2, 0, 3)
        
        # 3. 计算缩放点积注意力
        # Q: (batch, num_heads, seq_len, d_k)
        # K: (batch, num_heads, seq_len, d_k)
        # V: (batch, num_heads, seq_len, d_v)
        
        # 计算注意力分数
        # Q @ K^T: (batch, num_heads, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)
        
        # Softmax
        attention_weights = F.softmax(scores, dim=-1)
        
        # 加权求和
        # (batch, num_heads, seq_len, seq_len) @ (batch, num_heads, seq_len, d_v)
        # -> (batch, num_heads, seq_len, d_v)
        head_outputs = torch.matmul(attention_weights, V)
        
        # 4. 拼接所有头的输出
        # (batch, num_heads, seq_len, d_v) -> (batch, seq_len, num_heads, d_v)
        head_outputs = head_outputs.permute(0, 2, 1, 3)
        # (batch, seq_len, num_heads, d_v) -> (seq_len, batch, d_model)
        concat_output = head_outputs.contiguous().view(seq_len, batch_size, self.d_model)
        
        # 5. 最终线性层
        output = self.W_o(concat_output)
        
        return output


# 测试
print("=== 多头注意力测试 ===")
d_model = 4
num_heads = 2
seq_len = 5
batch_size = 3

# 创建模型
mha = MultiHeadAttention(d_model, num_heads)

# 随机输入
X = torch.randn(seq_len, batch_size, d_model)
output = mha(X)

print(f"输入形状: {X.shape}")
print(f"输出形状: {output.shape}")
print(f"输出是否与输入形状一致: {output.shape == X.shape}")

# 检查各头的维度
print(f"\n每个头的维度 d_k = d_v = {d_model // num_heads}")

# 验证参数数量
total_params = sum(p.numel() for p in mha.parameters())
print(f"总参数量: {total_params}")
print(f"预期参数量: 4 * {d_model} * {d_model} = {4 * d_model * d_model}")

=== 多头注意力测试 ===
输入形状: torch.Size([5, 3, 4])
输出形状: torch.Size([5, 3, 4])
输出是否与输入形状一致: True

每个头的维度 d_k = d_v = 2
总参数量: 64
预期参数量: 4 * 4 * 4 = 64
